# Chapter 11 assignment: identifying damping and stiffness from noisy displacements

*Companion notebook to Chapter 11 (Inverse Problems, Data Assimilation and Uncertainty Quantification).*
Runs on a CPU in a few minutes; no GPU is needed.

**Learning goals**

1. Write the loss of an inverse PINN with a hard-constrained trial solution and log-parametrized unknowns.
2. Compare the PINN estimate with classical output least squares (`solve_ivp` inside `least_squares`).
3. Quantify the uncertainty of the estimate with the Gauss-Newton covariance and with an ensemble trained on perturbed data.
4. Detect practical non-identifiability from the sensitivity matrix before training.

## Theory recap

The displacement of a damped oscillator satisfies
$$ y'' + c\,y' + k\,y = 0,\qquad y(0) = 1,\quad y'(0) = 0,\qquad t\in[0,10],$$
with unknown damping $c>0$ and stiffness $k>0$ (true values $c=0.4$, $k=4$). We observe $d_i = y(t_i) + \varepsilon_i$
at $N=25$ times with Gaussian noise of standard deviation $\sigma$.

* **Trial solution.** $y_\theta(t) = 1 + (t/T)^2 N_\theta(t)$ satisfies both initial conditions exactly (Chapter 3).
* **Log-parametrization.** $c = e^{\rho_1}$, $k = e^{\rho_2}$ keeps the parameters positive and makes the optimizer act on relative changes.
* **Loss.** $\mathcal{L}(\theta,\rho) = \frac1N\sum_i (y_\theta(t_i) - d_i)^2 + \lambda_r \frac1M \sum_j \bigl(R(t_j)/4\bigr)^2$ with the residual $R = y_\theta'' + c\,y_\theta' + k\,y_\theta$ (divided by the typical size $k\approx 4$ of its terms).
* **Linearized uncertainty.** If $J$ is the Jacobian of the model values with respect to $(c,k)$ at the least-squares estimate, the covariance is approximately $\sigma^2 (J^\top J)^{-1}$ (Section 11.6).
* **Discrepancy principle.** At a good estimate the RMS data misfit is close to $\sigma$.

In [ ]:
import numpy as np
import torch
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from pinnode import (MLP, ConstrainedModel, derivative, to_tensor, train, uniform, rel_l2,
                     configure_torch, set_seed)

configure_torch("float64", threads=1)
set_seed(2024)

C_TRUE, K_TRUE, T = 0.4, 4.0, 10.0

def exact(t, c=C_TRUE, k=K_TRUE):
    # underdamped solution with y(0) = 1, y'(0) = 0
    a = c / 2
    w = np.sqrt(k - a ** 2)
    return np.exp(-a * t) * (np.cos(w * t) + a / w * np.sin(w * t))

## Task 1 (given): synthetic data

Run the cell. Do not change the seed: the checks below assume these data.

In [ ]:
N, ETA = 25, 0.02
t_obs = np.linspace(0.0, T, N + 1)[1:]
rng = np.random.default_rng(7)
sigma = ETA * np.max(np.abs(exact(t_obs)))
d_obs = exact(t_obs) + sigma * rng.standard_normal(N)
print(f"noise standard deviation sigma = {sigma:.4f}")

## Task 2: the inverse PINN loss

Complete `loss_fn`. Use `derivative(y, t_col)` for $y'$ and `derivative(dy, t_col)` for $y''$.
The initial guess is $(c_0, k_0) = (1, 2)$.

In [ ]:
def make_pinn(seed=0, p0=(1.0, 2.0)):
    torch.manual_seed(seed)
    net = MLP(1, 1, width=32, depth=3, activation="tanh", domain=(0.0, T))
    model = ConstrainedModel(net, lambda t, n: 1.0 + (t / T) ** 2 * n)   # y(0)=1, y'(0)=0 exactly
    log_p = torch.nn.Parameter(torch.log(torch.tensor(p0)))
    return model, log_p

def make_loss(model, log_p, d, lam_r=1.0):
    t_col = to_tensor(uniform(200, 0.0, T), requires_grad=True)
    t_d, dd = to_tensor(t_obs), to_tensor(d)

    def loss_fn():
        c, k = torch.exp(log_p)
        y = model(t_col)
        # TODO: first and second derivatives of y with respect to t_col
        dy = ...
        d2y = ...
        residual = (d2y + c * dy + k * y) / 4.0
        # TODO: data misfit = mean of (model(t_d) - dd)**2
        L_d = ...
        L_r = (residual ** 2).mean()
        return L_d + lam_r * L_r, {"data": L_d, "residual": L_r}

    return loss_fn

## Task 3: train and check

Train with Adam followed by L-BFGS (about half a minute on one core).

In [ ]:
model, log_p = make_pinn(seed=0)
loss_fn = make_loss(model, log_p, d_obs)
hist = train(list(model.parameters()) + [log_p], loss_fn, adam_iters=3000, lr=5e-3,
             lbfgs_iters=500, verbose=False)
c_hat, k_hat = torch.exp(log_p).detach().numpy()
rms = float(np.sqrt(hist.parts["data"][-1]))
print(f"PINN estimate: c = {c_hat:.4f}, k = {k_hat:.4f};  RMS misfit = {rms:.4f} (sigma = {sigma:.4f})")

In [ ]:
# Check your answer
assert abs(c_hat / C_TRUE - 1) < 0.10, "c is more than 10 % off: check the residual and the derivatives"
assert abs(k_hat / K_TRUE - 1) < 0.02, "k is more than 2 % off"
assert 0.4 * sigma < rms < 1.5 * sigma, "the data misfit should be comparable with the noise level"
print("Task 3 OK")

## Task 4: classical output least squares

Write $y''+cy'+ky=0$ as a first-order system $z=(y,y')$ and return the model values at `t_obs`.

In [ ]:
def model_values(q):
    c, k = np.exp(q)
    # TODO: solve the first-order system with solve_ivp(method="DOP853", rtol=1e-10, atol=1e-12,
    #       t_eval=t_obs) on (0, T) from z(0) = (1, 0) and return the first component
    ...

res = least_squares(lambda q: model_values(q) - d_obs, np.log([1.0, 2.0]), diff_step=1e-6)
c_ls, k_ls = np.exp(res.x)
print(f"least squares: c = {c_ls:.4f}, k = {k_ls:.4f}")

In [ ]:
# Check your answer: both estimators should nearly coincide on the same data
assert abs(c_hat - c_ls) / c_ls < 0.05 and abs(k_hat - k_ls) / k_ls < 0.01
print("Task 4 OK")

## Task 5: Gauss-Newton standard deviations

`res.jac` is the Jacobian with respect to $(\log c, \log k)$; the chain rule converts it to $(c, k)$.

In [ ]:
J = res.jac / np.exp(res.x)[None, :]
# TODO: covariance sigma^2 (J^T J)^{-1} and the standard deviations of c and k
cov = ...
sd = ...
print(f"sd(c) = {sd[0]:.4f}, sd(k) = {sd[1]:.4f}")

In [ ]:
# Check your answer: the true values should lie within a few standard deviations
z = np.abs(np.array([c_ls - C_TRUE, k_ls - K_TRUE])) / sd
assert np.all(z < 4.0), z
assert 0.002 < sd[0] < 0.05 and 0.001 < sd[1] < 0.05, sd
print("Task 5 OK, z-scores:", np.round(z, 2))

## Task 6: an ensemble with perturbed data

Train $5$ PINNs, member $m$ on $d + \sigma\,\xi^{(m)}$ with $\xi^{(m)}\sim\mathcal{N}(0,I)$ and its own seed, and
compare the standard deviation of the members with the Gauss-Newton values. Then repeat with the *same* data for
all members (only the seed changes). Which ensemble reflects the data uncertainty, and why?
(About three minutes in total.)

In [ ]:
members_perturbed, members_seed_only = [], []
for m in range(5):
    xi = np.random.default_rng(100 + m).standard_normal(N)
    # TODO: train one PINN on d_obs + sigma * xi (seed 10 + m) and one on d_obs (seed 10 + m);
    #       append (c, k) of each to the corresponding list
    ...

members_perturbed, members_seed_only = np.array(members_perturbed), np.array(members_seed_only)
print("perturbed data: sd =", members_perturbed.std(axis=0, ddof=1))
print("seed only:      sd =", members_seed_only.std(axis=0, ddof=1))
print("Gauss-Newton:   sd =", sd)

## Task 7: practical identifiability before training

Compute the linearized relative standard deviations of $c$ and $k$ for observations at $25$ equally spaced times in
$(0, 10]$ and in $(0, 0.5]$ (same noise level), using finite-difference sensitivities of `exact`. Which parameter
becomes practically non-identifiable on the short window, and why? (Hint: how much of the decay $e^{-ct/2}$ and how much
of the curvature $y''\approx -ky$ near $t=0$ does the short window contain?)

In [ ]:
def sensitivities(t, c=C_TRUE, k=K_TRUE, h=1e-6):
    dc = (exact(t, c * (1 + h), k) - exact(t, c * (1 - h), k)) / (2 * c * h)
    dk = (exact(t, c, k * (1 + h)) - exact(t, c, k * (1 - h))) / (2 * k * h)
    return np.stack([dc, dk], axis=1)

for name, tt in (("(0, 10]", t_obs), ("(0, 0.5]", np.linspace(0.0, 0.5, N + 1)[1:])):
    S = sensitivities(tt)
    # TODO: relative standard deviations sigma * sqrt(diag((S^T S)^{-1})) / (c, k)
    rel_sd = ...
    print(name, np.round(rel_sd, 4))

## Reflection

1. Repeat Task 3 with $\lambda_r = 10^{-3}$ and $\lambda_r = 10^{2}$. Relate the RMS misfit to $\sigma$ (discrepancy principle).
2. Start Task 4 from $(c_0, k_0) = (1, 8)$. Does least squares still converge to the same estimate? Does the PINN?
3. Why can the PINN be trained without an ODE solver, and what does that buy for partially observed systems (Section 11.3)?